## Further K Exploration **Draft**
We are comparing both models at a fixed parameter of $k = 322$, which provides a controlled baseline, Spectral Clustering deomnstrates a structural advantage, particularly in the broader recommendation brackets (maintaining higher Top-5 and Top-10 scores than Louvain). However, forcing Spectral Clustering to match the dynamic community count discovered by Louvain introduces a significant methological constraint. Louvain optimizes for network modularity and density, naturally settling on 322 communities as its ideal structural representation of the data. Spectral Clustering relies on identifying optimal mathematical cuts within the graph's eigenspace. 

By artificially restricting Spectral to 322 partitions, we are likely handicapping its underlying logic and forcing distinct semantic micro-communities to remain merged. This leads us to hypothesize that 322 is not the optimal parameter for Spectral in this high-dimensional space. To evaluate its true predicitve potential, we must decouple its parameterization from Louvain's density-based results, mathematically determine Spectral's own optimal number of clusters, and analyze its unconstrained performance. 

#### The Search of the Optimal K (Silhouettte and Eigengap Heuristics)
In tuning Spectral Clustering, we have had to pick $k$ as a hyperparameter, this $k$ has not been arbitrarily chosen. We started by calculating the *silhouette/eigengap* score for every $k$ up to 322, which was the optimal $k$ found by Louvain, seeing as the score kept increasing the interval was brought up to 1500, with sampling at every 50th $k$, the optimal $k$ was found at 700. To more accurately pin-point the exact $k$ we calculated silhouette scores of surrounding $k$s (675-725) finding $k = 720$ to be optimal. 

While we previously utilized a **Delta WCSS** threshold to determine $k=55$ for representative-based models, graph-based models requires a different form of validation in choosing the optimal hyperparameter $k$. 

- Eigengap Heuristic: We analyze the eigenvalues $\lambda$ of the graph Laplacian to find the largest difference between consecutive values, defined as $\delta_k = \lambda_{k+1} - \lambda_k$. A maximized gap indicates the number of stable, natural communities within the network density. 

- Average Silhouette Coefficient: Because spectral clustering projects the sparse TF-IDF data into a new, low-dimensional coordinate system, we use the silhouette coefficient to evaluate the quality of the resulting clusters within the **spectral embedding space**. 

- Davies-Bouldin Index (DBI): To provide a final check on the 'efficiency' of our high-granularity partition, we utilize the DBI. This metric identifies the average similarity between each cluster $i$ and its most similar neighbor $j$, where $s$ is the cluster diameter and $d_{ij}$ is the distance between centroids:

$$DBI = \frac{1}{k}\sum_{i=1}^kmax_{j \neq i}(\frac{s_i+s_j}{d_{ij}})$$

For any individual playlist $i$, the silhouette scores $s(i)$ is calculated by compaing its average distance to other points in the same cluster, i.e. its cohesion $a(i)$ against its average distance to points in the nearest neighboring cluster, i.e. its separation $b(i)$:

$$s(i) = \frac{b(i) - a(i)}{max(a(i), b(i))}$$

The means of these individual scores across the entire contextual dataset is used in guiding the selection of $k$ because of these following properties:
- A high average silhouette scores (approaching 1) indicated that the spectral transformation has successfully mapped the complex, arbitrary shapes of user-generated musical contexts into well-separated, compact clusters in the eigenvector space. 
- While the eigengap heuristic identifies candidate values for $k$ based on the graph's connectivity, the silhouette coefficient provides a geometric 'sanity check' to ensure these partitions are cohesive. 
- Since human musical curation is often nested or high-dimensional, scores near 0 alert us to overlapping contexts where $k$ may be too high, preventing the 'gravity well' effect previously observed in the K-Means implementation

By calculating this metric within the spectral embedding space, we can verify if the graph-based network representation offers a higher 'semantic purity' than our previous representative-based baseline.

In [ ]:
# Displaying of the silhouette scores, for different k values
path_img_1 = "scripts/data/spectral_k_analysis_spotify_0_322.png"
path_img_2 = "scripts/data/spectral_k_analysis_spotify_0_1500.png"
path_img_3 = "scripts/data/spectral_k_analysis_spotify_OPT720.png"

# 1x3 subplot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(plt.imread(path_img_1))
axes[0].set_title("Spectral K Analysis (0-322)")
axes[0].axis('off')
axes[1].imshow(plt.imread(path_img_2))
axes[1].set_title("Spectral K Analysis (0-1500)")
axes[1].axis('off')
axes[2].imshow(plt.imread(path_img_3))
axes[2].set_title("Spectral K Analysis (675-725)")
axes[2].axis('off')
plt.tight_layout()
plt.show()

The optimal $k$ was found to be 720, based on both the silhouette score maximized and a strong supporting DBI score, which was relatively low. They are not in agreement with eigengap which states that the optimal $k$ would be 55.

#### Executing Spectral Clustering - $k = 720$

In [ ]:
# Initialize Spectral Clustering instance, and load the graph from the shared builder
from graph.knn.spectral_clustering import SpectralGraphClustering
Spectral_720 = SpectralGraphClustering(graph=graph_builder.G, n_clusters=720, graph_config_name=graph_config_name)

In [ ]:
execution_pipeline(Spectral_720, df, unique_texts, tfidf_matrix)

### Evaluation
#### Spectral Clustering Evaluation

In [ ]:
spectral_report_path_720 = evaluation_pipeline(Spectral_720, df, unique_texts, tfidf_matrix)

#### Spectral vs. Louvain

In [ ]:
# Printing the two graphs side by side for comparison
if os.path.exists(spectral_report_path_720) and os.path.exists(louvain_report_out):
    print("\n[INFO] Displaying evaluation comparison graphs for Spectral and Louvain...")
    
    spectral_f01 = os.path.join(spectral_report_path_720, "f01_comparison.png")
    louvain_f01 = os.path.join(louvain_report_out, "f01_comparison.png")
    
    if os.path.exists(spectral_f01) and os.path.exists(louvain_f01):
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        axes[0].imshow(plt.imread(spectral_f01))
        axes[0].set_title(f"{Spectral_720.algo_name} F0.1 Comparison")
        axes[0].axis('off')
        
        axes[1].imshow(plt.imread(louvain_f01))
        axes[1].set_title(f"{Louvain.algo_name} F0.1 Comparison")
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        print("[WARNING] One or both F0.1 comparison graphs are missing. Cannot display side by side.")